# Project: Complete ML Pipeline with Hyperparameter Tuning

Build an end-to-end ML pipeline with preprocessing, cross-validation, and hyperparameter tuning using GridSearchCV and RandomizedSearchCV.

In [ ]:
import sys, os, subprocess
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_PATH = '/content/drive/MyDrive/data-science-mastery'
    if os.path.isdir(REPO_PATH):
        os.chdir(REPO_PATH)
        print(f'Working directory: {os.getcwd()}')
        if not os.path.isdir('Data') or len(os.listdir('Data')) < 5:
            subprocess.check_call([sys.executable, 'scripts/prepare_datasets.py'])
    else:
        REPO_PATH = '/content/data-science-mastery'
        if os.path.isdir(REPO_PATH):
            os.chdir(REPO_PATH)
        else:
            print('Repo not found. Run opencolab_setup.ipynb first.')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge, Lasso
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
print('Libraries loaded')

In [ ]:
np.random.seed(42)
n = 2000
df = pd.DataFrame({
    'sqft': np.random.normal(2000, 500, n).astype(int),
    'bedrooms': np.random.randint(2, 6, n),
    'bathrooms': np.random.randint(1, 4, n),
    'year_built': np.random.randint(1950, 2024, n),
    'location': np.random.choice(['Downtown','Suburb','Rural','Waterfront'], n),
    'garage': np.random.choice([0,1,2,3], n, p=[0.2,0.4,0.3,0.1]),
})
df['price'] = (df['sqft']*120 + df['bedrooms']*8000 + df['bathrooms']*10000 +
               (2024-df['year_built'])*(-500) + df['garage']*15000 +
               (df['location']=='Downtown')*50000 + (df['location']=='Waterfront')*80000 +
               (df['location']=='Suburb')*20000 + np.random.normal(0,30000,n)).astype(int)
print(f'Dataset: {df.shape}')
print(f'Price: ${df["price"].min():,.0f} - ${df["price"].max():,.0f}')
df.head()

## Build Preprocessing and Model Pipeline

In [ ]:
num_feats = ['sqft','bedrooms','bathrooms','year_built','garage']
cat_feats = ['location']
preprocessor = ColumnTransformer([
    ('num', Pipeline([('imputer',SimpleImputer(strategy='median')), ('scaler',StandardScaler())]), num_feats),
    ('cat', Pipeline([('imputer',SimpleImputer(strategy='constant',fill_value='Unknown')),
                      ('onehot',OneHotEncoder(handle_unknown='ignore'))]), cat_feats),
])
X, y = df.drop('price', axis=1), df['price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print('Pipeline built. Train:', X_train.shape[0], 'Test:', X_test.shape[0])

## Compare Multiple Models

In [ ]:
models = {
    'Ridge': Ridge(random_state=42),
    'Lasso': Lasso(random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
}
results = []
for name, model in models.items():
    pipe = Pipeline([('prep', preprocessor), ('reg', model)])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    results.append({'Model': name, 'R2': r2_score(y_test, y_pred),
                    'RMSE': np.sqrt(mean_squared_error(y_test, y_pred)),
                    'MAE': mean_absolute_error(y_test, y_pred)})
print(pd.DataFrame(results).round(2).to_string(index=False))

## Hyperparameter Tuning

In [ ]:
pipe = Pipeline([('prep', preprocessor), ('reg', RandomForestRegressor(random_state=42))])
params = {
    'reg__n_estimators': [100, 200, 300],
    'reg__max_depth': [10, 20, 30, None],
    'reg__min_samples_split': [2, 5, 10],
}
search = RandomizedSearchCV(pipe, params, n_iter=15, cv=3, scoring='r2', random_state=42, verbose=0)
search.fit(X_train, y_train)
print('Best params:', search.best_params_)
print(f'Best CV R2: {search.best_score_:.4f}')
y_pred = search.predict(X_test)
print(f'Test R2: {r2_score(y_test, y_pred):.4f}')

## Learning Curves

In [ ]:
from sklearn.model_selection import learning_curve
train_sizes, train_scores, test_scores = learning_curve(
    search.best_estimator_, X_train, y_train, cv=3, n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 10), scoring='r2')
plt.figure(figsize=(10, 5))
plt.plot(train_sizes, train_scores.mean(axis=1), 'o-', label='Train', color='blue')
plt.plot(train_sizes, test_scores.mean(axis=1), 'o-', label='Validation', color='red')
plt.fill_between(train_sizes, train_scores.mean(axis=1)-train_scores.std(axis=1),
                 train_scores.mean(axis=1)+train_scores.std(axis=1), alpha=0.1, color='blue')
plt.fill_between(train_sizes, test_scores.mean(axis=1)-test_scores.std(axis=1),
                 test_scores.mean(axis=1)+test_scores.std(axis=1), alpha=0.1, color='red')
plt.xlabel('Training Set Size')
plt.ylabel('R2 Score')
plt.title('Learning Curves')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print(f'Final test R2 with best model: {r2_score(y_test, search.predict(X_test)):.4f}')

## Summary

- Built preprocessing pipeline (numeric + categorical)
- Compared Ridge, Lasso, RF, Gradient Boosting
- Hyperparameter tuning with RandomizedSearchCV
- Learning curves for bias-variance analysis